# Task 2 — Data Preprocessing and Regression Modeling

**Dataset:** `data/raw/sample_50000.csv`  
**Target:** `fare_amount`  
**Random split:** 80/20, `random_state=42`  
**Leakage control:** data-dependent transformations are fitted inside scikit-learn pipelines and CV folds.

This notebook documents the workflow. The reproducible training entry point is `python -m src.uber_fare.train`. Do not report metrics until that command finishes and writes the results files.

## 1. Cleaning decisions

- Convert source coordinate columns from radians to degrees.
- Remove rows with invalid/non-positive fares or unparseable timestamps.
- Remove trips whose pickup or drop-off coordinates fall outside the documented NYC bounding box, including `(0, 0)` placeholders.
- Remove passenger counts outside 1–6 and report the count.
- Recompute trip and reference-point distances from coordinates; do not use the supplied distance columns as model inputs.
- Do not remove valid high fares only because they exceed an IQR threshold; inspect them first.

These are deterministic rules based on data validity. Imputers, encoders and scalers are fitted only inside the training pipeline.

In [6]:
from pathlib import Path
import sys
import pandas as pd

# Find the project folder containing src/uber_fare
current = Path.cwd()
candidates = [current, *current.parents]

PROJECT_ROOT = next(
    (
        p for p in candidates
        if (p / "src" / "uber_fare").is_dir()
    ),
    None
)

# Also support running from the analytics_project/notebooks folder
if PROJECT_ROOT is None:
    for p in candidates:
        if (p / "analytics_project" / "src" / "uber_fare").is_dir():
            PROJECT_ROOT = p / "analytics_project"
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate src/uber_fare. "
        "Open Jupyter from the project folder and check the folder structure."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.uber_fare.data import load_clean_data

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "sample_50000.csv"

df_clean = load_clean_data(DATA_PATH)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Rows after cleaning:", len(df_clean))
print("Missing values:", int(df_clean.isna().sum().sum()))
print("Non-positive fares:", int((df_clean["fare_amount"] <= 0).sum()))
print(
    "Passenger counts outside 1–6:",
    int((~df_clean["passenger_count"].between(1, 6)).sum())
)

df_clean.head()


Project root: C:\Users\noura\RouteIQ_Uber_Fare_Prediction_Final\analytics_project
Dataset path: C:\Users\noura\RouteIQ_Uber_Fare_Prediction_Final\analytics_project\data\raw\sample_50000.csv
Rows after cleaning: 48897
Missing values: 0
Non-positive fares: 0
Passenger counts outside 1–6: 159


,User ID,User Name,Driver Name,Car Condition,Weather,Traffic Condition,key,fare_amount,pickup_datetime,pickup_longitude,...,month,weekday,year,jfk_dist,ewr_dist,lga_dist,sol_dist,nyc_dist,distance,bearing
0,KHVrEVlD,Kimberly Adams,Amy Butler,Very Good,windy,Congested Traffic,2009-06-15 17:26:21,4.5,2009-06-15 17:26:21,-73.844311,...,6,0,2009,20.265840,55.176046,14.342611,34.543548,27.572573,1.030764,-2.918897
1,lPxIuEri,Justin Tapia,Hannah Zimmerman,Excellent,cloudy,Flow Traffic,2010-01-05 16:52:16,16.9,2010-01-05 16:52:16,-74.016048,...,1,1,2010,44.667679,31.832358,23.130775,15.125872,8.755732,8.450134,-0.375217
2,gsVN8JLS,Elizabeth Lopez,Amanda Jackson,Bad,stormy,Congested Traffic,2011-08-18 00:35:00,5.7,2011-08-18 00:35:00,-73.982738,...,8,3,2011,43.597686,33.712082,19.865289,17.722624,9.847344,1.389525,2.599961
3,9I7kWFgd,Steven Wilson,Amy Horn,Very Good,stormy,Flow Traffic,2012-04-21 04:30:42,7.7,2012-04-21 04:30:42,-73.987130,...,4,5,2012,42.642965,32.556289,21.063132,15.738963,7.703421,2.799270,0.133905
4,8QN5ZaGN,Alexander Andrews,Cassandra Larson,Bad,stormy,Congested Traffic,2010-03-09 07:51:00,5.3,2010-03-09 07:51:00,-73.968095,...,3,1,2010,43.329953,39.406828,15.219339,23.732406,15.600745,1.999157,-0.502703


## 2. Split before learned preprocessing

The final hold-out test set is not used for feature ablation or hyperparameter selection. Because fare patterns drift over years, a separate chronological robustness check is also reported.

In [7]:
from sklearn.model_selection import train_test_split

model_df = df_clean.loc[df_clean.passenger_count.between(1, 6)].copy()
RAW_COLS = ["pickup_datetime", "pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude", "passenger_count", "Car Condition", "Weather", "Traffic Condition"]
X = model_df[RAW_COLS]
y = model_df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print({"train_rows": len(X_train), "test_rows": len(X_test)})

{'train_rows': 38990, 'test_rows': 9748}


## 3. Pipeline design

`UberFeatureEngineer` accepts raw degree-based coordinates and pickup datetime, then derives trip distance, distances to documented reference coordinates, cyclic time features, and year. Numeric imputation/scaling and categorical imputation/one-hot encoding are contained in the model pipeline. Trees do not require scaling; the Ridge baseline uses `RobustScaler`.

In [8]:
from src.uber_fare.features import UberFeatureEngineer, haversine_km
from src.uber_fare.train import make_pipeline
from sklearn.linear_model import Ridge

example = X_train.head(3).copy()
engineered = UberFeatureEngineer(include_categoricals=True).fit_transform(example)
engineered.head()

,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,passenger_count,distance_km,jfk_dist_km,ewr_dist_km,lga_dist_km,sol_dist_km,...,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos,year,Car Condition,Weather,Traffic Condition
19049,40.751282,-73.994053,40.751670,-73.990797,1,0.277644,21.932134,16.686904,10.504436,8.107343,...,-0.500000,-0.866025,0.000000,1.000000,8.660254e-01,-0.5,2009,Bad,windy,Dense Traffic
45893,40.768374,-73.982285,40.767264,-73.993169,1,0.924868,22.268800,18.418938,9.167703,10.246465,...,-0.866025,0.500000,-0.433884,-0.900969,-8.660254e-01,-0.5,2010,Excellent,windy,Dense Traffic
28115,40.761868,-73.986438,40.738569,-73.985421,1,2.592150,22.094842,17.773630,9.615133,9.446221,...,-0.500000,0.866025,0.433884,-0.900969,1.224647e-16,-1.0,2013,Very Good,stormy,Congested Traffic


## 4. Feature ablation and model comparison

The training script compares Ridge with and without `Car Condition`, `Weather`, and `Traffic Condition` using the same shuffled 5-fold CV on training data. The lower mean CV RMSE determines whether the categorical fields are retained. It then evaluates Linear/Ridge baseline, Decision Tree, Random Forest, and HistGradientBoosting, and tunes Random Forest with `RandomizedSearchCV` using training folds only. The hold-out test metrics are reported in dollars (MAE, RMSE) plus R².

Run from the repository root:

```bash
python -m src.uber_fare.train
```

Expected outputs after a successful run: `reports/categorical_ablation.csv`, `reports/model_comparison.csv`, `reports/chronological_check.json`, `models/uber_fare_pipeline_v1.joblib`, and `models/model_metadata.json`.

## 5. Results and interpretation

Results are intentionally not hard-coded into this notebook. Populate/inspect the generated CSV and JSON files only after the training command finishes successfully. Compare CV results (used for model selection) separately from test results (final hold-out estimate). Do not select a model solely because it has the lowest training error.

### What I would try next

After validating the baseline results, investigate the chronological 2015 hold-out, alternative fare-target transforms, robust evaluation of zero-distance trips, and permutation importance. Keep these experiments separate from the locked test-set decision.